# Bronze Layer – Raw Streaming Ingestion (Healthcare Lakehouse)

The **Bronze Layer** is responsible for ingesting raw healthcare data from external source files into the lakehouse in near real time.

This layer preserves source fidelity while applying only minimal transformations required to ensure stable ingestion.

Bronze tables serve as the **trusted landing zone** for downstream Silver standardization and Gold business analytics.

---

# Objectives of the Bronze Layer

- Ingest raw provider, patient, claim, policy, and payment data
- Preserve original data structure and lineage
- Handle schema drift and late-arriving records safely
- Standardize minimal datatypes required for storage
- Capture ingestion metadata for auditability

---

# Bronze Streams by Business Domain

## Master Data Streams

These datasets provide the foundational entities used across all use cases.

### Providers Stream (CSV)
- Ingests hospital/doctor information
- Casts provider_id to numeric
- Adds ingestion timestamp and source tracking
- Used later for provider analytics and fraud monitoring

### Payers Stream (CSV)
- Ingests insurance company data
- Standardizes payer identifiers
- Supports reconciliation and payment analysis

### Patients Stream (JSON)
- Ingests patient demographic records
- Handles multiple DOB formats safely
- Ensures ingestion never fails due to date parsing
- Supports fraud behaviour analysis and claim mapping

### Policies Stream (Nested JSON)
- Ingests insurance policy structure
- Extracts nested patient, payer, and coverage fields
- Standardizes coverage limit and copay values
- Supports adjudication logic in Gold layer

---

## Transactional Claim Streams

These datasets represent the operational claim lifecycle.

### Claims Stream (CSV)
- Ingests claim headers
- Handles mixed date formats safely
- Standardizes financial values
- Serves as the primary claim identity table

### Claim Lines Stream (Nested JSON)
- Ingests detailed procedures per claim
- Flattens nested arrays safely
- Ensures each medical service is captured separately
- Enables accurate billing aggregation for decision, fraud, and reconciliation

### Payments Stream (JSON)
- Ingests insurer payment records
- Standardizes payment identifiers and amounts
- Captures payment ingestion timestamp
- Supports settlement tracking in Gold layer

---

# What Bronze Transformations Are Doing

Across all streams, Bronze applies consistent ingestion safeguards:

### Schema Evolution Mode (Rescue)
Ensures new or unexpected columns do not break ingestion.

### Safe Casting
Numeric identifiers and amounts are standardized early to prevent downstream type issues.

### Flexible Date Parsing
Handles multiple real-world date formats so streaming never fails.

### Nested JSON Handling
Safely extracts structured fields from complex payloads.

### Metadata Enrichment
Each record receives:
- ingestion_ts → when data entered the platform
- source_system → original file source for lineage tracking

---

# Why Bronze Is Critical for Business Use Cases

Although Bronze is not used directly by dashboards, it enables:

- Accurate claim totals for adjudication
- Reliable payment records for reconciliation
- Clean provider/patient mapping for fraud checks
- Traceable audit lineage for compliance

Without a stable Bronze layer, Silver standardization and Gold analytics would not be reliable.

---

# Business Value of the Bronze Layer

-Prevents ingestion failures from schema drift  
-Captures raw truth of healthcare transactions  
-Ensures replayability of historical files  
-Maintains audit trail for compliance  
-Provides consistent foundation for Silver cleansing  

---

The Bronze layer ensures that **all healthcare events enter the platform reliably, traceably, and without data loss**, forming the base of the lakehouse architecture.


In [0]:
# %sql
# CREATE VOLUME IF NOT EXISTS healthcare.storage.raw;
# CREATE VOLUME IF NOT EXISTS healthcare.storage.chk;
# CREATE VOLUME IF NOT EXISTS healthcare.storage.schema;


In [0]:
# CONFIGURATION

stream_bronze_db = "healthcare.stream_bronze"
raw_base = "/Volumes/healthcare/storage/raw/"
chk_base = "/Volumes/healthcare/storage/chk/"
schema_base = "/Volumes/healthcare/storage/schema/"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {stream_bronze_db}")


In [0]:
# ---- RESET AUTO LOADER STATE FOR ALL BRONZE STREAMS ----

paths = [
    "provider",
    "patient",
    "claim",
    "claim_line",
    "payment",
    "policy",
    "payer"
]

for p in paths:
    dbutils.fs.rm(schema_base + p, True)
    dbutils.fs.rm(chk_base + p, True)

print("All Bronze checkpoints & schemas cleared")


In [0]:
# %sql
# DROP TABLE IF EXISTS healthcare.stream_bronze.provider;
# DROP TABLE IF EXISTS healthcare.stream_bronze.payer;
# DROP TABLE IF EXISTS healthcare.stream_bronze.patient;
# DROP TABLE IF EXISTS healthcare.stream_bronze.policy;
# DROP TABLE IF EXISTS healthcare.stream_bronze.claim;
# DROP TABLE IF EXISTS healthcare.stream_bronze.claim_line;
# DROP TABLE IF EXISTS healthcare.stream_bronze.payment;


In [0]:
# dbutils.fs.rm("/Volumes/healthcare/storage/chk/", True)
# dbutils.fs.rm("/Volumes/healthcare/storage/schema/", True)

# dbutils.fs.mkdirs("/Volumes/healthcare/storage/chk/")
# dbutils.fs.mkdirs("/Volumes/healthcare/storage/schema/")


In [0]:
from pyspark.sql.functions import col, current_timestamp, lit

providers_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")

    # avoid datatype surprises from CSV inference
    .option("cloudFiles.inferColumnTypes", "false")

    # allow new columns without breaking stream
    .option("cloudFiles.schemaEvolutionMode", "rescue")

    # Auto Loader tracking
    .option("cloudFiles.schemaLocation", schema_base + "provider")

    .load(raw_base + "providers/")
)

# ---- STANDARDIZE ----
providers_stream_clean = providers_stream.select(
    col("provider_id").cast("long"),
    col("provider_name"),
    col("specialty"),
    col("state"),
    current_timestamp().alias("ingestion_ts"),
    lit("providers_file").alias("source_system")
)

# ---- WRITE TO BRONZE ----
(
    providers_stream_clean.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", chk_base + "provider")
    .trigger(availableNow=True)
    .toTable(f"{stream_bronze_db}.provider")
)


In [0]:
from pyspark.sql.functions import col, current_timestamp, lit

payers_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")

    # prevents datatype surprises from CSV inference
    .option("cloudFiles.inferColumnTypes", "false")

    # allows new columns without failing stream
    .option("cloudFiles.schemaEvolutionMode", "rescue")

    # Auto Loader state tracking
    .option("cloudFiles.schemaLocation", schema_base + "payer")

    .load(raw_base + "payers/")
)

# ---- STANDARDIZE ----
payers_stream_clean = payers_stream.select(
    col("payer_id").cast("long"),
    col("payer_name"),
    col("plan_type"),
    current_timestamp().alias("ingestion_ts"),
    lit("payers_file").alias("source_system")
)

# ---- WRITE TO BRONZE ----
(
    payers_stream_clean.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", chk_base + "payer")
    .trigger(availableNow=True)
    .toTable(f"{stream_bronze_db}.payer")
)


In [0]:
from pyspark.sql.functions import col, current_timestamp, lit, expr, coalesce

patients_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")

    # prevents stream failure if JSON changes
    .option("cloudFiles.schemaEvolutionMode", "rescue")

    # Auto Loader tracking
    .option("cloudFiles.schemaLocation", schema_base + "patients")

    .load(raw_base + "patients/")
)

# ---- SAFE STANDARDIZATION ----
patients_stream_clean = patients_stream.select(
    col("patient_id").cast("long"),
    col("name").alias("patient_name"),

    # FINAL SAFE DOB PARSING (never crashes stream)
    coalesce(
        expr("try_to_date(dob, 'yyyy-MM-dd')"),
        expr("try_to_date(dob, 'dd-MM-yyyy')"),
        expr("try_to_date(dob, 'MM/dd/yyyy')")
    ).alias("dob"),

    col("gender"),
    col("state"),

    current_timestamp().alias("ingestion_ts"),
    lit("patients_file").alias("source_system")
)

# ---- WRITE TO BRONZE ----
(
    patients_stream_clean.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", chk_base + "patients")
    .trigger(availableNow=True)
    .toTable(f"{stream_bronze_db}.patient")
)


In [0]:
from pyspark.sql.types import *

policy_schema = StructType([
    StructField("policy_id", LongType(), True),
    StructField("member", StructType([
        StructField("patient_id", LongType(), True)
    ]), True),
    StructField("payer", StructType([
        StructField("payer_id", LongType(), True)
    ]), True),
    StructField("coverage", StructType([
        StructField("plan", StringType(), True),
        StructField("limit", DoubleType(), True),
        StructField("copay", DoubleType(), True)
    ]), True)
])


In [0]:
from pyspark.sql.functions import col, current_timestamp, lit

policies_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")

    # prevents stream failure if JSON structure changes
    .option("cloudFiles.schemaEvolutionMode", "rescue")

    # Auto Loader tracking
    .option("cloudFiles.schemaLocation", schema_base + "policy")

    # explicit schema required
    .schema(policy_schema)

    .load(raw_base + "policies_nested/")
)

# ---- SAFE EXTRACTION FROM NESTED JSON ----
policies_stream_clean = policies_stream.select(
    col("policy_id").cast("long"),

    # nested safe extraction
    col("member.patient_id").cast("long").alias("patient_id"),
    col("payer.payer_id").cast("long").alias("payer_id"),

    col("coverage.plan").alias("plan"),
    col("coverage.limit").cast("double").alias("coverage_limit"),
    col("coverage.copay").cast("double").alias("copay"),

    # Bronze audit fields
    current_timestamp().alias("ingestion_ts"),
    lit("policies_file").alias("source_system")
)

# ---- WRITE TO BRONZE ----
(
    policies_stream_clean.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", chk_base + "policy")
    .trigger(availableNow=True)
    .toTable(f"{stream_bronze_db}.policy")
)


In [0]:
from pyspark.sql.functions import col, current_timestamp, lit, expr, coalesce

# ---- READ STREAM ----
claims_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")

    # prevents wrong datatype inference
    .option("cloudFiles.inferColumnTypes", "false")

    # schema tracking
    .option("cloudFiles.schemaLocation", schema_base + "claim")

    .load(raw_base + "claims/")
)

# ---- CLEAN + SAFE DATE PARSING ----
claims_stream_clean = claims_stream.select(
    col("claim_id").cast("long"),
    col("provider_id").cast("long"),
    col("patient_id").cast("long"),
    col("policy_id").cast("long"),

    # FINAL SAFE DATE PARSER (never crashes)
    coalesce(
        expr("try_to_date(claim_date, 'yyyy-MM-dd')"),
        expr("try_to_date(claim_date, 'dd-MM-yyyy')")
    ).alias("claim_date"),

    col("claim_amount").cast("double"),
    current_timestamp().alias("ingestion_ts"),
    lit("claims_file").alias("source_system")
)

# ---- WRITE STREAM ----
(
    claims_stream_clean.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", chk_base + "claim")
    .trigger(availableNow=True)
    .toTable(f"{stream_bronze_db}.claim")
)


In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import col, explode_outer, current_timestamp, lit

# ---- SCHEMA ----
claim_lines_schema = StructType([
    StructField("claim_id", LongType(), True),
    StructField("procedures", ArrayType(
        StructType([
            StructField("code", StringType(), True),
            StructField("charge", DoubleType(), True)
        ])
    ), True)
])

# ---- READ STREAM ----
claim_lines_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")

    # Keeps malformed JSON instead of failing stream
    .option("cloudFiles.schemaEvolutionMode", "rescue")

    # Required for Auto Loader tracking
    .option("cloudFiles.schemaLocation", schema_base + "claim_line")

    # Explicit schema = stable ingestion
    .schema(claim_lines_schema)

    .load(raw_base + "claim_lines/")
)

# ---- FLATTEN SAFELY ----
claim_lines_stream = claim_lines_stream.select(
    col("claim_id"),
    explode_outer(col("procedures")).alias("proc")   #prevents null explode failure
)

# ---- STANDARDIZE ----
claim_lines_stream = claim_lines_stream.select(
    col("claim_id").cast("long"),
    col("proc.code").alias("procedure_code"),
    col("proc.charge").cast("double").alias("charge"),
    current_timestamp().alias("ingestion_ts"),
    lit("claim_lines_file").alias("source_system")
)

# ---- WRITE TO BRONZE ----
(
    claim_lines_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", chk_base + "claim_line")
    .trigger(availableNow=True)
    .toTable(f"{stream_bronze_db}.claim_line")
)


In [0]:
from pyspark.sql.types import *

payment_schema = StructType([
    StructField("payment_id", LongType(), True),
    StructField("claim_id", LongType(), True),
    StructField("payer_id", LongType(), True),
    StructField("amount", DoubleType(), True),
    StructField("status", StringType(), True)
])


In [0]:
from pyspark.sql.functions import col, current_timestamp, lit

payments_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")

    #keeps malformed JSON instead of failing
    .option("cloudFiles.schemaEvolutionMode", "rescue")

    #Auto Loader tracking
    .option("cloudFiles.schemaLocation", schema_base + "payment")

    #explicit schema (required)
    .schema(payment_schema)

    .load(raw_base + "payments/")
)

# ---- STANDARDIZE ----
payments_stream_clean = payments_stream.select(
    col("payment_id").cast("long"),
    col("claim_id").cast("long"),
    col("payer_id").cast("long"),
    col("amount").cast("double"),   # safe numeric cast
    col("status"),
    current_timestamp().alias("ingestion_ts"),
    lit("payments_file").alias("source_system")
)

# ---- WRITE TO BRONZE ----
(
    payments_stream_clean.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", chk_base + "payment")
    .trigger(availableNow=True)
    .toTable(f"{stream_bronze_db}.payment")
)


In [0]:
%sql
SELECT 'provider' AS table, COUNT(*) FROM healthcare.stream_bronze.provider
UNION ALL
SELECT 'payer', COUNT(*) FROM healthcare.stream_bronze.payer
UNION ALL
SELECT 'patient', COUNT(*) FROM healthcare.stream_bronze.patient
UNION ALL
SELECT 'policy', COUNT(*) FROM healthcare.stream_bronze.policy
UNION ALL
SELECT 'claim', COUNT(*) FROM healthcare.stream_bronze.claim
UNION ALL
SELECT 'claim_line', COUNT(*) FROM healthcare.stream_bronze.claim_line
UNION ALL
SELECT 'payment', COUNT(*) FROM healthcare.stream_bronze.payment;


In [0]:
%sql
SELECT * FROM healthcare.stream_bronze.provider where provider_name='Manaswin' --8001


In [0]:
%sql
SELECT * FROM healthcare.stream_bronze.claim where claim_id='10003'--70001
